# 12 — Décision de publication Hugging Face

**Objectif :** auditer les artefacts sans les publier automatiquement.

**Entrées :** manifeste local et décision de source.  
**Sortie :** `reports/huggingface_decision.json`.  
**Dépendance :** notebook 11.  
**Temps estimé :** moins d'une minute.  
**Ressources :** CPU, aucune connexion requise en dry-run.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Racine du projet introuvable.")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.publish_hf import collect_publishable_files

ARTIFACTS_DIR = ROOT / "artifacts"
REPORTS_DIR = ROOT / "reports"
inventory = json.loads((REPORTS_DIR / "data_inventory.json").read_text(encoding="utf-8"))
manifest = json.loads((ARTIFACTS_DIR / "manifest.json").read_text(encoding="utf-8"))
files = collect_publishable_files(ARTIFACTS_DIR)

source = inventory["source"]
license_confirmed = bool(manifest["metadata"].get("redistribution_allowed", False))
should_publish = source == "kaggle" and license_confirmed

if source != "kaggle":
    reason = "Artefacts produits à partir du corpus synthétique : publication sans intérêt."
elif not license_confirmed:
    reason = "Droits de redistribution des artefacts dérivés non confirmés."
else:
    reason = "Bundle entraîné sur la source réelle et licence explicitement confirmée."

decision = {
    "publish": should_publish,
    "repository_visibility": "private",
    "source": source,
    "license_confirmed": license_confirmed,
    "candidate_files": [path.name for path in files],
    "total_bytes": sum(path.stat().st_size for path in files),
    "reason": reason,
    "message": (
        "Publication Hugging Face justifiée après confirmation utilisateur."
        if should_publish
        else "Aucun artefact ne nécessite actuellement une publication sur Hugging Face."
    ),
}
(REPORTS_DIR / "huggingface_decision.json").write_text(
    json.dumps(decision, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(decision, indent=2, ensure_ascii=False))

## Commande future, volontairement inactive

```powershell
$env:HF_TOKEN="..."  # ne jamais committer cette valeur
.\.venv\Scripts\python.exe -m src.publish_hf `
  --repo-id BADR-JOULAlI/xbox-search-recommender `
  --private --confirm-license --push
```

Sans `--push` et `--confirm-license`, le script effectue seulement un dry-run.